# Chấm metric B1 — notebook tự tìm hoặc clone repository

Notebook đọc B1 đã có trong repository, không sinh lại B1. Nếu mở riêng trên Colab/Kaggle, notebook tự clone repository qua `REPO_URL`/`REPO_BRANCH`. Notebook đọc checkpoint đã có và chỉ chạy metric còn thiếu. Mặc định bật đường full để cài/chấm BERTScore, COMET và BLEURT; 16 metric nhẹ đã có sẽ skip. Metric lỗi được ghi lại và không làm Run All dừng.

In [ ]:
from pathlib import Path
import base64
import csv
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
import zlib
B1_FALLBACK_B64 = "..."

# Tự tìm work/ hoặc submit/ từ vị trí notebook. Nếu notebook được mở riêng
# trên Colab/Kaggle, tự clone repository rồi mới tìm artifact.
HERE = Path.cwd().resolve()
REPO_URL = os.environ.get(
    "META_JUDGE_REPO_URL",
    "https://github.com/thanhnghi-do-2k3/llm-as-judge.git",
)
REPO_BRANCH = os.environ.get("META_JUDGE_REPO_BRANCH", "main")
CLONE_DIR = Path(
    os.environ.get("META_JUDGE_CLONE_DIR", str(HERE / "llm-as-judge"))
).expanduser().resolve()


def is_work_root(path):
    return (
        (path / "meta-judge" / "src" / "vn_meta_judge").is_dir()
        and (path / "resources" / "source.csv").is_file()
    )


def is_submit_root(path):
    return (
        (path / "meta-judge" / "src" / "vn_meta_judge").is_dir()
        and (path / "resources" / "source.csv").is_file()
        and (path / "analysis" / "b1_chrf_scores.csv").is_file()
    )


ROOT = None
candidate_paths = []
for candidate in (HERE, *HERE.parents):
    # In a local checkout Jupyter often starts in the project root while the
    # runnable artifacts live below work/ or submit/. Check those directories
    # before falling back to a network clone.
    for nested in (candidate, candidate / "work", candidate / "submit"):
        if nested not in candidate_paths:
            candidate_paths.append(nested)
for candidate in candidate_paths:
    if is_submit_root(candidate):
        ROOT = candidate
        break
    if is_work_root(candidate):
        ROOT = candidate
        break
if ROOT is None:
    print(f"Không thấy repository local; clone {REPO_URL} -> {CLONE_DIR}")
    if not CLONE_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(CLONE_DIR)],
            check=True,
        )
    if not is_work_root(CLONE_DIR):
        raise FileNotFoundError(
            f"Repository clone không đủ resources/meta-judge: {CLONE_DIR}"
        )
    ROOT = CLONE_DIR

CODE_DIR = ROOT / "meta-judge"
SOURCE_FILE = ROOT / "resources" / "source.csv"
if (ROOT / "analysis" / "b1_chrf_scores.csv").is_file():
    B1_CSV_FILE = ROOT / "analysis" / "b1_chrf_scores.csv"
    OUTPUT_DIR = ROOT / "analysis"
elif (ROOT / "cheat-runtime" / "b1_chrf_scores.csv").is_file():
    B1_CSV_FILE = ROOT / "cheat-runtime" / "b1_chrf_scores.csv"
    OUTPUT_DIR = ROOT / "cheat-runtime" / "metric-output" / "b1"
else:
    # Bản B1 nén kèm notebook là fallback để notebook riêng vẫn chạy nếu
    # repository remote chưa được cập nhật artifact mới nhất.
    B1_CSV_FILE = ROOT / "cheat-runtime" / "b1_chrf_scores.csv"
    B1_CSV_FILE.parent.mkdir(parents=True, exist_ok=True)
    B1_CSV_FILE.write_bytes(zlib.decompress(base64.b64decode(B1_FALLBACK_B64)))
    OUTPUT_DIR = ROOT / "cheat-runtime" / "metric-output" / "b1"

SYNTHETIC_FILE = OUTPUT_DIR / "b1_rule_based.jsonl"
NORMALIZED_CSV = OUTPUT_DIR / "b1_rule_based_normalized.csv"
RUN_HEAVY_METRICS = True        # Chạy metric nặng còn thiếu; metric đã cache sẽ skip.
RUN_UNDERTHESEA = False         # True sẽ thêm BLEU/chrF underthesea.
INSTALL_DEPENDENCIES = True
LIGHT_FAMILIES = ["BLEU", "chrF", "ROUGE", "METEOR"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not B1_CSV_FILE.is_file():
    raise FileNotFoundError(f"Thiếu B1 artifact: {B1_CSV_FILE}")
if not SOURCE_FILE.is_file():
    raise FileNotFoundError(f"Thiếu source/reference: {SOURCE_FILE}")
if not (CODE_DIR / "src" / "vn_meta_judge").is_dir():
    raise FileNotFoundError(f"Thiếu mã metric: {CODE_DIR}")

required_modules = {
    "pandas": "pandas",
    "evaluate": "evaluate",
    "sacrebleu": "sacrebleu",
    "rouge_score": "rouge-score",
    "nltk": "nltk",
}
missing = [package for module, package in required_modules.items()
           if importlib.util.find_spec(module) is None]
if missing and INSTALL_DEPENDENCIES:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *missing],
            check=True,
        )
    except Exception as exc:
        print(f"CẢNH BÁO: không cài được package metric nhẹ {missing}: {exc}")
missing_after_install = [
    package for module, package in required_modules.items()
    if importlib.util.find_spec(module) is None
]
if missing_after_install:
    print(
        "CẢNH BÁO: vẫn thiếu package metric nhẹ; các metric tương ứng sẽ được "
        "ghi vào errors và notebook vẫn hoàn tất: ", missing_after_install
    )

sys.path.insert(0, str(CODE_DIR / "src"))
CACHE_ROOT = ROOT / "cache" / "b1"
os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["NLTK_DATA"] = str(CACHE_ROOT / "nltk")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
HEAVY_FAMILY_REQUIREMENTS = {
    "BERTScore": CODE_DIR / "requirements-metric-bertscore.txt",
    "COMET": CODE_DIR / "requirements-metric-comet.txt",
    "BLEURT": CODE_DIR / "requirements-metric-bleurt.txt",
}
FAMILY_PACKAGE_PATHS = {}
HEAVY_AVAILABLE = False
if RUN_HEAVY_METRICS:
    for family, requirement in HEAVY_FAMILY_REQUIREMENTS.items():
        package_dir = CACHE_ROOT / "metric-packages" / family.lower()
        marker = package_dir / (
            ".ready-" + hashlib.sha256(requirement.read_bytes()).hexdigest()[:16]
        )
        try:
            package_dir.mkdir(parents=True, exist_ok=True)
            if not marker.is_file():
                print(f"Cài package metric nặng: {family}")
                subprocess.run(
                    [
                        sys.executable, "-m", "pip", "install", "-q",
                        "--upgrade", "--no-deps", "--target", str(package_dir),
                        "-r", str(requirement),
                    ],
                    check=True,
                )
                marker.write_text("ready\n", encoding="utf-8")
            FAMILY_PACKAGE_PATHS[family] = package_dir
            sys.path.insert(0, str(package_dir))
        except Exception as exc:
            print(f"CẢNH BÁO: không cài được {family}; sẽ bỏ qua family này: {exc}")
    HEAVY_AVAILABLE = bool(FAMILY_PACKAGE_PATHS)
    if len(FAMILY_PACKAGE_PATHS) != len(HEAVY_FAMILY_REQUIREMENTS):
        print("Một số package metric nặng thiếu; các family cài được vẫn tiếp tục chạy.")
PREFETCH_MARKER = CACHE_ROOT / ".light-metrics-ready"
if not PREFETCH_MARKER.is_file():
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    prefetch_env = os.environ.copy()
    prefetch_env["PYTHONPATH"] = str(CODE_DIR / "src")
    try:
        subprocess.run(
            [sys.executable, "-m", "vn_meta_judge.notebook_workflow", "prefetch"],
            cwd=CODE_DIR,
            env=prefetch_env,
            check=True,
        )
        PREFETCH_MARKER.write_text("ready\n", encoding="utf-8")
    except Exception as exc:
        print(f"CẢNH BÁO: prefetch không hoàn tất; metric nào thiếu dữ liệu sẽ được ghi lỗi: {exc}")
print("ROOT:", ROOT)
print("B1:", B1_CSV_FILE)
print("Output:", OUTPUT_DIR)
ACTIVE_HEAVY = RUN_HEAVY_METRICS and HEAVY_AVAILABLE
ACTIVE_HEAVY_FAMILIES = list(FAMILY_PACKAGE_PATHS)
print("Mode:", "light + optional underthesea", "| heavy:", ACTIVE_HEAVY)


## Chuẩn hóa input

Ghép source/reference từ `resources/source.csv`, kiểm tra đủ 300 câu × 6 mức và ghi JSONL trung gian. Không gọi API.

In [ ]:
from collections import Counter

with SOURCE_FILE.open(encoding="utf-8-sig", newline="") as handle:
    source_rows = {str(row["ID_cau_VLSP"]): row for row in csv.DictReader(handle)}
with B1_CSV_FILE.open(encoding="utf-8-sig", newline="") as handle:
    b1_rows = list(csv.DictReader(handle))

required = {"id", "level", "operation", "prediction_vi"}
if not b1_rows or not required.issubset(b1_rows[0]):
    raise ValueError(f"B1 CSV thiếu cột: {sorted(required)}")

normalized = []
for row in b1_rows:
    row_id = str(row["id"]).strip()
    if row_id not in source_rows:
        raise ValueError(f"B1 id không có trong source.csv: {row_id}")
    level = int(row["level"])
    if level not in range(6):
        raise ValueError(f"B1 level ngoài 0..5: {level}")
    source = source_rows[row_id]
    normalized.append({
        "id": row_id,
        "source_zh": source["Nguon_ZH"].strip(),
        "reference_vi": source["Tham_chieu_VI"].strip(),
        "prediction_vi": row["prediction_vi"].strip(),
        "damage_level": level,
        "prompt_type": "rule_based",
        "model_name": "deterministic-rule-baseline-precomputed",
        "backend": "analysis_csv",
        "temperature": 0.0,
        "operation": row["operation"].strip(),
        "status": "ok",
    })

keys = {(row["id"], row["damage_level"]) for row in normalized}
if len(normalized) != 1800 or len(keys) != 1800:
    raise ValueError(f"B1 phải có 1.800 khóa duy nhất, nhận được {len(normalized)}")
for row_id in source_rows:
    levels = sorted(row["damage_level"] for row in normalized if row["id"] == row_id)
    if levels != list(range(6)):
        raise ValueError(f"B1 thiếu level cho {row_id}: {levels}")

with SYNTHETIC_FILE.open("w", encoding="utf-8") as handle:
    for row in normalized:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
with NORMALIZED_CSV.open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(normalized[0]))
    writer.writeheader()
    writer.writerows(normalized)
print("Rows:", len(normalized), "| operations:", Counter(r["operation"] for r in normalized))


## Chấm và xuất CSV

Metric runner dùng cache checkpoint theo thư mục output; chạy lại notebook sẽ tái sử dụng kết quả hợp lệ.

In [ ]:
import pandas as pd
try:
    # COMET 2.2.x unpacks XLM-R outputs as (hidden, pooler, all_hidden),
    # while recent Transformers returns (hidden, all_hidden) when COMET
    # disables the pooling layer.  Patch the adapter before loading models.
    from comet.encoders import xlmr as _comet_xlmr

    def _notebook_xlmr_forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=False,
        )
        last_hidden_states = outputs[0]
        all_layers = outputs[-1]
        return {
            "sentemb": last_hidden_states[:, 0, :],
            "wordemb": last_hidden_states,
            "all_layers": all_layers,
            "attention_mask": attention_mask,
        }

    _comet_xlmr.XLMREncoder.forward = _notebook_xlmr_forward
except Exception as exc:
    print(f"CẢNH BÁO: không cài được COMET XLM-R compatibility patch: {exc}")

import metrics.hf_metrics as _hf_metrics
from vn_meta_judge.metrics.runner import run_metric_scores

# The notebook may clone an older repository revision.  Install the COMET
# adapter in this kernel as well as in the repository source so all checkpoint
# schemas work without requiring a remote Git update.
def _notebook_comet(ref, pred, src=None, model_name="Unbabel/wmt22-comet-da", device=None):
    from comet import load_from_checkpoint
    from vn_meta_judge.metrics.comet_cache import download_comet_checkpoint

    device = device or _hf_metrics.REWARD_DEVICE
    cache_key = f"comet_{model_name}"
    if cache_key not in _hf_metrics.CACHED_METRICS:
        checkpoint = download_comet_checkpoint(model_name)
        _hf_metrics.CACHED_METRICS[cache_key] = load_from_checkpoint(checkpoint)
    metric = _hf_metrics.CACHED_METRICS[cache_key]
    src = src if src is not None else [""] * len(ref)
    values = {"src": src, "mt": pred, "ref": ref}
    input_segments = getattr(getattr(metric, "hparams", None), "input_segments", None)
    if input_segments:
        batches = [
            {name: values[name][index] for name in input_segments if name in values}
            for index in range(len(pred))
        ]
    elif "qe" in model_name.lower():
        batches = [{"src": s, "mt": p} for s, p in zip(src, pred)]
    else:
        batches = [{"src": s, "mt": p, "ref": r} for s, p, r in zip(src, pred, ref)]

    gpu_idx = _hf_metrics._parse_cuda_index(device)
    if gpu_idx is not None:
        predict_kwargs = {"accelerator": "gpu", "devices": [gpu_idx], "gpus": 1}
    elif device == "cuda":
        predict_kwargs = {"accelerator": "gpu", "gpus": 1}
    else:
        predict_kwargs = {"accelerator": "cpu", "gpus": 0}
    with _hf_metrics._cuda_device_guard():
        output = metric.predict(
            batches, batch_size=32, progress_bar=False, **predict_kwargs
        )
    scores = output["scores"] if isinstance(output, dict) else output.scores
    return list(scores)

_hf_metrics.METRIC_FNS["comet"] = _notebook_comet

# The direct runner expects BLEURT checkpoints to have been extracted into the
# local HF cache.  Prepare them once before scoring so a fresh Kaggle/Colab
# runtime does not silently leave all BLEURT columns missing.
if ACTIVE_HEAVY and "BLEURT" in ACTIVE_HEAVY_FAMILIES:
    from vn_meta_judge.config import paper_metric_specs
    from vn_meta_judge.metrics.shard import _prepare_heavy_asset

    for spec in paper_metric_specs():
        if spec.family == "BLEURT":
            try:
                _prepare_heavy_asset(spec.key)
            except Exception as exc:
                print(f"CẢNH BÁO: không chuẩn bị được {spec.key}: {exc}")

tokenizations = ["syllable"] + (["underthesea"] if RUN_UNDERTHESEA else [])
scores_by_tokenization = {}
for tokenization in tokenizations:
    families = ["BLEU", "chrF"] if tokenization == "underthesea" else LIGHT_FAMILIES
    if ACTIVE_HEAVY and tokenization == "syllable":
        families = families + ACTIVE_HEAVY_FAMILIES
    scores_path = OUTPUT_DIR / f"scores_b1_{tokenization}.json"
    existing = {}
    if scores_path.is_file():
        try:
            existing = json.loads(scores_path.read_text(encoding="utf-8")).get("scores", {})
        except json.JSONDecodeError:
            print("Checkpoint hỏng; metric runner sẽ tạo lại:", scores_path)
    print(tokenization, "đã có", len(existing), "metric; chỉ chạy phần còn thiếu.")
    try:
        result = run_metric_scores(
            SYNTHETIC_FILE,
            scores_path,
            tokenization=tokenization,
            include_bleurt="BLEURT" in families,
            selected_families=families,
            batch_size=256,
        )
    except Exception as exc:
        # A missing optional package/model must not kill Run All. Preserve any
        # checkpoint that was written before the runner failed and make the
        # failure visible in the same JSON used for resume/retry.
        print(f"CẢNH BÁO {tokenization}: metric runner không khởi động được: {exc}")
        payload = {}
        if scores_path.is_file():
            try:
                payload = json.loads(scores_path.read_text(encoding="utf-8"))
            except json.JSONDecodeError:
                payload = {}
        payload.setdefault("scores", {})
        payload.setdefault("errors", {})
        payload["errors"]["__runner__"] = repr(exc)
        scores_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
        result = {"scores": payload["scores"], "failed_metrics": len(families) * 4}
    if result.get("failed_metrics"):
        payload = json.loads(scores_path.read_text(encoding="utf-8"))
        print(
            f"CẢNH BÁO {tokenization}: {result['failed_metrics']} metric chưa chạy được; "
            "đã giữ checkpoint thành công và tiếp tục."
        )
        for key, error in payload.get("errors", {}).items():
            print(" -", key, ":", error)
    scores_by_tokenization[tokenization] = result
    rows = [json.loads(line) for line in SYNTHETIC_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
    payload = json.loads(scores_path.read_text(encoding="utf-8"))
    frame = pd.DataFrame(rows)
    for key, values in payload["scores"].items():
        frame[f"metric__{key}"] = values
    output_csv = OUTPUT_DIR / f"b1_metric_scores_{tokenization}.csv"
    frame.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(tokenization, "rows=", len(frame), "metrics=", len(payload["scores"]), "->", output_csv)


## Kiểm tra cuối

In [ ]:
import pandas as pd

csv_path = OUTPUT_DIR / "b1_metric_scores_syllable.csv"
frame = pd.read_csv(csv_path)
assert len(frame) == 1800
assert sorted(frame["damage_level"].unique().tolist()) == list(range(6))
metric_columns = [column for column in frame.columns if column.startswith("metric__")]
if not metric_columns:
    print("CẢNH BÁO: chưa có metric nào hoàn tất; kiểm tra log rồi chạy lại notebook.")

archived = pd.read_csv(B1_CSV_FILE)
archived_columns = [column for column in archived.columns if column.startswith("chrf_")]
max_archived_diff = 0.0
for column in archived_columns:
    scored_column = f"metric__{column}"
    if scored_column in frame:
        max_archived_diff = max(
            max_archived_diff,
            float((frame[scored_column] - archived[column]).abs().max()),
        )
print("Verified rows:", len(frame))
print("Metric columns:", len(metric_columns))
expected_metric_count = 16 + (4 * len(ACTIVE_HEAVY_FAMILIES) if ACTIVE_HEAVY else 0)
print("Expected metric columns:", expected_metric_count)
if len(metric_columns) < expected_metric_count:
    print("CẢNH BÁO: còn thiếu", expected_metric_count - len(metric_columns), "metric; xem errors trong scores_b1_syllable.json và chạy lại.")
print("Archived chrF max diff:", max_archived_diff)
print("Final CSV:", csv_path)
